In [1]:
# 1. Import required libraries
import os
import re
import pandas as pd
import numpy as np
import plotly.graph_objects as go


# 2. Set the folder containing the SR-LAB files
data_folder = r'C:\Users\lsang\OneDrive\Universiteit\Neuroscience and Cognition\Profile Research Project\03_Analysis\02_Behaviour\PPI\PPI_columns'


# 3. Functions to load the SR-LAB files
def ColumNames():
    mouse_n = [1, 2, 3, 4] * 72
    trial_n = ['Trial' + str(i) for i in range(1, 73) for _ in range(4)]
    return [
        trial_n[i] + '_Mouse' + str(mouse_n[i])
        for i in range(288)
    ]


def LoadNewSRLabFile(file_path):
    raw = pd.read_excel(file_path, header=None)

    row_lookup = {
        str(raw.iloc[i, 0]).strip(): i
        for i in range(len(raw))
    }

    required = ['Chamber_ID', 'Subject_ID', 'Trial_ID', 'Trial_Num']
    missing = [name for name in required if name not in row_lookup]

    if missing:
        raise ValueError(f'Missing required rows: {missing}')

    chamber_row = row_lookup['Chamber_ID']
    subject_row = row_lookup['Subject_ID']
    condition_row = row_lookup['Trial_ID']
    trial_num_row = row_lookup['Trial_Num']

    data_rows = [
        i for i in range(len(raw))
        if str(raw.iloc[i, 0]).strip().startswith('Data')
    ]
    data_rows = sorted(
        data_rows,
        key=lambda i: int(str(raw.iloc[i, 0]).strip()[4:])
    )

    converted = {}

    for col in range(1, raw.shape[1]):
        subject = raw.iloc[subject_row, col]

        if pd.isna(subject) or str(subject).strip() == '':
            continue

        chamber = raw.iloc[chamber_row, col]
        trial_num = raw.iloc[trial_num_row, col]
        condition = raw.iloc[condition_row, col]

        if pd.isna(chamber) or pd.isna(trial_num) or pd.isna(condition):
            continue

        chamber = int(chamber)
        trial_num = int(trial_num)

        if isinstance(subject, (float, np.floating)) and float(subject).is_integer():
            subject = str(int(subject))
        else:
            subject = str(subject).strip()

        trial_name = f'Trial{trial_num}_Mouse{chamber}_{subject}'

        values = [np.nan] * 8
        values[2] = subject
        values[5] = str(condition).strip()
        values[6] = chamber
        values[7] = trial_num

        values.extend(
            pd.to_numeric(raw.iloc[data_rows, col], errors='coerce').tolist()
        )

        converted[trial_name] = values

    return pd.DataFrame(converted)


def LoadSRLabFile(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    if extension in ['.xlsx', '.xls']:
        return LoadNewSRLabFile(file_path)

    big_df = pd.read_csv(file_path)
    big_df.columns = ColumNames()
    return big_df


# 4. Ask for the animal ID
animal_id = input('Enter the animal ID: ').strip()


# 5. Load files and identify relevant trials
files = [
    f for f in os.listdir(data_folder)
    if f.lower().endswith(('.csv', '.xlsx', '.xls'))
]

found_trials = 0

for file in files:

    file_path = os.path.join(data_folder, file)

    try:
        df = LoadSRLabFile(file_path)

    except Exception as e:
        print(f'Could not load {file}: {e}')
        continue

    # Find trials belonging to this animal.
    # In the current Excel format, the animal ID is stored in row 2.
    animal_trials = []

    for trial in df.columns:
        trial_id = str(df.loc[2, trial]).strip()

        if trial_id.endswith('.0'):
            trial_id = trial_id[:-2]

        if trial_id == animal_id:
            animal_trials.append(trial)

    # 6. Plot each relevant trial
    for trial in animal_trials:

        condition = df.loc[5, trial]

        waveform = pd.to_numeric(
            df[trial].iloc[8:],
            errors='coerce'
        )

        # Preserve the original sample positions as milliseconds
        time = waveform.index - 8

        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=time,
            y=waveform,
            mode='lines',
            name=trial
        ))

        fig.update_layout(
            title=f'Animal {animal_id} | {file} | {trial} | {condition}',
            xaxis_title='Time (ms)',
            yaxis_title='Startle response',
            #yaxis_range=[0, 3000],
            template='plotly_white'
        )

        fig.show()
        found_trials += 1


# 7. Report results
if found_trials == 0:
    print(f'No trials found for animal {animal_id}.')
else:
    print(f'\nDisplayed {found_trials} trials for animal {animal_id}.')


Displayed 72 trials for animal A16.
